# Create Gene List

In [5]:
%cd /content/drive/MyDrive/Github/NLP

/content/drive/MyDrive/Github/NLP


In [6]:
import pandas as pd

In [3]:
genedf = pd.read_excel('GeneNames.xlsx')

In [4]:
genedf['genelist'] = None
for index, row in genedf.iterrows():
    try:
        AS = row['Approved symbol'].split(',')
    except:
        AS = []
    try:
        parts = row['Approved name'].split(',')
        max_length = max(len(part.strip()) for part in parts)
        AN = [part.strip() for part in parts if len(part.strip()) == max_length]
    except:
        AN = []
    try:
        PS = row['Previous symbols'].split(',')
    except:
        PS = []
    try:
        AS2 = row['Alias symbols'].split(',')
    except:
        AS2 = []

    # Combine lists and remove duplicates
    combined_list = list(set(AS + AN + PS + AS2))

    # Set the 'genelist' column in the DataFrame
    genedf.at[index, 'genelist'] = combined_list

In [ ]:
genedf

In [ ]:
geneslist = genedf.loc[genedf['Status']=='Approved']['genelist'].tolist()

In [ ]:
len(geneslist)

43619

In [ ]:
genedf['genelist'][1]

In [ ]:
flat_geneslist = [item.strip() for sublist in geneslist for item in sublist]

In [ ]:
flat_geneslist

# Select PMIDs

In [ ]:
import pandas as pd

In [8]:
#create PMID list
%cd /content/drive/MyDrive/Github/PubMedBERT-NER-Gene
df_49article = pd.read_excel('ConvertToOfficial.xlsx')
df_49article = df_49article[df_49article['related to DKD?'].notna()]
list_pmid = df_49article['pmid'].tolist()

/content/drive/MyDrive/Github/PubMedBERT-NER-Gene


In [ ]:
pmid_df = pd.read_csv('/content/drive/MyDrive/Github/PubMedBERT-NER-Gene/280paper.txt')
list_pmid = pmid_df['pmid'].tolist()

In [6]:
len(list_pmid)

55

# Create List 1

In [ ]:
import pandas as pd
from collections import defaultdict

In [ ]:
genedf['genelist']

In [ ]:
df_filtered = pd.read_pickle('/content/drive/MyDrive/Github/PubMedBERT-NER-Gene/df_filtered.pkl')

In [ ]:
def extract_approved_symbols(df_filtered, genedf, list_pmid):
    pmid_to_genes = defaultdict(set)
    for _, row in df_filtered.iterrows():
        list_gene = row['Gene']
        pmid = row['pmid']
        if pmid in list_pmid:
            for gene in list_gene:
                # Check if the gene exists in any row of genedf['genelist']
                matched_genes = genedf[genedf['genelist'].apply(lambda x: gene in x)]
                if not matched_genes.empty:
                    approved_symbol = matched_genes.iloc[0]['Approved symbol']
                    pmid_to_genes[pmid].add(approved_symbol)  # Use a set to avoid duplicates

    # Convert the dictionary to a DataFrame
    final_df = pd.DataFrame({
        'pmid': list(pmid_to_genes.keys()),
        'approved_symbols': [list(genes) for genes in pmid_to_genes.values()]
    })

    return final_df

In [ ]:
final_df = extract_approved_symbols(df_filtered, genedf, list_pmid)

In [ ]:
final_df

In [ ]:
final_df.loc[final_df['pmid'] == 25428125]

,pmid,approved_symbols
33,25428125,[PRKAA2]


In [ ]:
df_49article['pmid'] = df_49article['pmid'].astype(str)
final_df['pmid'] = final_df['pmid'].astype(str)

df_49article_with_symbols = pd.merge(df_49article, final_df, on='pmid', how='left')

df_49article_with_symbols['approved_symbols'] = df_49article_with_symbols['approved_symbols'].apply(lambda x: x if isinstance(x, list) else [])

In [ ]:
df_49article_with_symbols.loc[df_49article_with_symbols['pmid'] == '25428125']

,pmid,title,abstract,nGene,Official DKD.related.gene,related to DKD?,number of genes,The thing that was found is a gene?,how many gene was found?,approved_symbols
13,25428125,AMP-activated protein kinase (AMPK) activation...,Diabetic nephropathy is characterized by diffu...,AMPK,PRKAA2,y,2.0,y,NaN,[PRKAA2]


In [ ]:
df_49article_with_symbols.to_csv('/content/drive/MyDrive/Github/PubMedBERT-NER-Gene/49paper_genes.csv')

In [ ]:
test_list = final_df['gene'].drop_duplicates().tolist()

In [ ]:
list_df = pd.DataFrame({'gene': test_list})

In [ ]:
list_df.to_csv('/content/drive/MyDrive/Github/PubMedBERT-NER-Gene/280paper_genes_removed_duplicates.csv')

# Use mygene API

In [ ]:
!pip install mygene

In [8]:
import pandas as pd
import mygene
from collections import defaultdict
from urllib.parse import quote
import re

In [7]:
df_filtered = pd.read_pickle('/content/drive/MyDrive/Github/PubMedBERT-NER-Gene/df_filtered.pkl')

In [20]:
df_filtered = df_filtered.loc[df_filtered['pmid'] == 25611208]

In [55]:
df_filtered

,index,pmid,title,abstract,label_biobert,Gene
2965,8771,25446993,Novel curcumin analog C66 prevents diabetic ne...,Glomerulosclerosis and interstitial fibrosis r...,1,"[connective tissue growth factor, CTGF, plasmi..."


In [ ]:
def normalize_gene_names(gene_list):
    mg = mygene.MyGeneInfo()
    normalized_genes = []

    for gene in gene_list:
        result = mg.query(gene, fields="symbol")
        if result and 'hits' in result and len(result['hits']) > 0:
            normalized_genes.append(result['hits'][0]['symbol'])
        else:
            normalized_genes.append(gene)  # If no match, keep the original name

    return normalized_genes

In [ ]:
gene_list = df_filtered['Gene'][0]
normalized_gene_list = normalize_gene_names(gene_list)
print(normalized_gene_list)

['DPP4', 'DPP-4I', 'GIP', 'INS', 'DPP4I', 'DPP4I', 'ACE2', 'ENOS', 'DPP4I', 'DPP4I', 'ALB', 'Wt', 'PDPN', 'IV', 'acta2.L', 'DPP4I', 'DPP4', 'IV', 'PDPN', 'DPP4', 'ENOS', 'DPP4']


In [ ]:
len(df_filtered['Gene'][0])

In [ ]:
len(normalized_gene_list)

In [47]:
df_filtered.loc[df_filtered['pmid'] == 25611208]

,index,pmid,title,abstract,label_biobert,Gene,Gene_normalized
2939,8695,25611208,"Empagliflozin, an Inhibitor of Sodium-Glucose ...",Advanced glycation end products (AGEs) and rec...,1,"[RAGE, sodium-glucose cotransporter 2, SGLT2, ...","[rage, sodium-glucose cotransporter 2, sglt2, ..."


In [ ]:
def extract_approved_symbols(df_filtered, list_pmid):
    mg = mygene.MyGeneInfo()
    pmid_to_genes = defaultdict(set)

    for _, row in df_filtered.iterrows():
        list_gene = row['Gene']
        pmid = row['pmid']
        if pmid in list_pmid:
            for gene in list_gene:
                # Encode the gene name
                encoded_gene = quote(gene)
                try:
                    # Query MyGene.info for the gene symbol
                    result = mg.query(encoded_gene, fields="symbol")
                    if result and 'hits' in result and len(result['hits']) > 0:
                        # Check if 'symbol' exists in the result
                        approved_symbol = result['hits'][0].get('symbol')
                        if approved_symbol:  # Only add if the symbol is found
                            pmid_to_genes[pmid].add(approved_symbol)
                except Exception as e:
                    print(f"Error querying gene {gene}: {e}")

    # Convert the dictionary to a DataFrame
    final_df = pd.DataFrame({
        'pmid': list(pmid_to_genes.keys()),
        'approved_symbols': [list(genes) for genes in pmid_to_genes.values()]
    })

    return final_df

In [ ]:
final_df= extract_approved_symbols(df_filtered, list_pmid)

In [ ]:
def preprocess_gene_name(gene_name):
    # Replace slashes and other problematic characters with spaces
    gene_name = re.sub(r'[\/]', ' ', gene_name)
    # Remove other problematic characters
    gene_name = re.sub(r'[^a-zA-Z0-9\s\-]', '', gene_name)
    return gene_name

def extract_approved_symbols(df_filtered, list_pmid):
    mg = mygene.MyGeneInfo()
    pmid_to_genes = defaultdict(set)

    for _, row in df_filtered.iterrows():
        list_gene = row['Gene']
        pmid = row['pmid']
        if pmid in list_pmid:
            for gene in list_gene:
                # Preprocess the gene name
                preprocessed_gene = preprocess_gene_name(gene)
                try:
                    # Query MyGene.info for the gene symbol
                    result = mg.query(preprocessed_gene, fields="symbol")
                    if result and 'hits' in result and len(result['hits']) > 0:
                        # Check if 'symbol' exists in the result
                        approved_symbol = result['hits'][0].get('symbol')
                        if approved_symbol:  # Only add if the symbol is found
                            pmid_to_genes[pmid].add(approved_symbol)
                except Exception as e:
                    print(f"Error querying gene {gene}: {e}")

    # Convert the dictionary to a DataFrame
    final_df = pd.DataFrame({
        'pmid': list(pmid_to_genes.keys()),
        'approved_symbols': [list(genes) for genes in pmid_to_genes.values()]
    })

    return final_df

In [ ]:
final_df2 = extract_approved_symbols(df_filtered, list_pmid)

In [ ]:
final_df

In [ ]:
final_df2

In [ ]:
df_49article['pmid'] = df_49article['pmid'].astype(str)
final_df2['pmid'] = final_df2['pmid'].astype(str)

df_49article_with_symbols = pd.merge(df_49article, final_df2, on='pmid', how='left')

df_49article_with_symbols['approved_symbols'] = df_49article_with_symbols['approved_symbols'].apply(lambda x: x if isinstance(x, list) else [])

In [ ]:
df_49article_with_symbols.to_csv('/content/drive/MyDrive/Github/PubMedBERT-NER-Gene/49paper_genes_New.csv')

In [ ]:
def preprocess_gene_name(gene_name):
    if pd.isna(gene_name):
        return ''
    # Replace slashes and other problematic characters with spaces
    gene_name = re.sub(r'[\/]', ' ', str(gene_name))
    # Remove other problematic characters
    gene_name = re.sub(r'[^a-zA-Z0-9\s\-]', '', gene_name)
    # Convert to lowercase
    gene_name = gene_name.lower()
    return gene_name

def preprocess_genedf(genedf):
    for idx, row in genedf.iterrows():
        genelist = row['genelist']
        if isinstance(genelist, list):
            genedf.at[idx, 'genelist'] = [preprocess_gene_name(g) for g in genelist]
        else:
            genedf.at[idx, 'genelist'] = []
        genedf.at[idx, 'Approved symbol'] = preprocess_gene_name(row['Approved symbol'])
    return genedf

def extract_approved_symbols(df_filtered, genedf, list_pmid):
    mg = mygene.MyGeneInfo()
    pmid_to_genes = defaultdict(set)

    # Preprocess genedf and df_filtered
    genedf = preprocess_genedf(genedf)
    df_filtered['Gene_normalized'] = df_filtered['Gene'].apply(lambda x: [preprocess_gene_name(g) for g in x])

    for _, row in df_filtered.iterrows():
        list_gene = row['Gene_normalized']
        pmid = row['pmid']
        if pmid in list_pmid:
            for gene in list_gene:
                # Query MyGene.info for the gene symbol
                try:
                    result = mg.query(gene, species='human', size=1)
                    if result and 'hits' in result and len(result['hits']) > 0:
                        # Extract the symbol from the result
                        symbol = result['hits'][0].get('symbol', '').lower()  # Extract and normalize symbol
                        if symbol:
                            # Check if the symbol exists in genedf['genelist']
                            matched_genes = genedf[genedf['genelist'].apply(lambda x: symbol in x)]
                            if not matched_genes.empty:
                                approved_symbol = matched_genes.iloc[0]['Approved symbol']
                                pmid_to_genes[pmid].add(approved_symbol.upper())
                    # No need to add the original gene name if no match found in genedf
                except Exception as e:
                    print(f"Error querying gene {gene}: {e}")

    # Convert the dictionary to a DataFrame
    final_df = pd.DataFrame({
        'pmid': list(pmid_to_genes.keys()),
        'approved_symbols': [list(genes) for genes in pmid_to_genes.values()]
    })

    return final_df

In [ ]:
final_df = extract_approved_symbols(df_filtered,genedf, list_pmid)

In [ ]:
final_df

,pmid,approved_symbols
0,26056446,"[TLR4, AKR1B1, AKR1B10]"
1,26046968,"[IGFL2, IGFBP2, PAPPA2, PAEP, IGFBP4]"
2,26022136,"[SLC25A33, CYBB, EPHX2, MAP3K2, MT-CO2, SOD1, ..."
3,26001596,"[IL17RE, CD4, MACORIS, MROCKI, IFNG, IL17A, IL..."
4,25977232,"[AGER, MOK, DSPP, DPP4, ILK]"
5,25971352,"[SOD3, ADIPOR2, SZT2, ALB]"
6,25969863,"[ISG20, TNFAIP8, CD69, CELIAC2, IK, CD4, LEISA..."
7,25967095,"[ALB, NOSTRIN, STK25, STK11, SIK1, CAT, PRKAB1]"
8,25963357,"[CRP, FTMT, TNFAIP8, CRIP3, IL6, LEISA1, HAMP,..."
9,25958041,"[CCDC88B, CASP12, BCL2L10, FABP4, EBP, BCL6B]"


In [ ]:
df_49article['pmid'] = df_49article['pmid'].astype(str)
final_df['pmid'] = final_df['pmid'].astype(str)

df_49article_with_symbols = pd.merge(df_49article, final_df, on='pmid', how='left')

df_49article_with_symbols['approved_symbols'] = df_49article_with_symbols['approved_symbols'].apply(lambda x: x if isinstance(x, list) else [])

In [ ]:
df_49article_with_symbols.to_csv('/content/drive/MyDrive/Github/PubMedBERT-NER-Gene/49paper_genes_New.csv')

## Check Genes and Apprroved Symboles

In [60]:
def preprocess_gene_name(gene_name):
    if pd.isna(gene_name):
        return ''
    # Replace slashes and other problematic characters with spaces
    gene_name = re.sub(r'[\/]', ' ', str(gene_name))
    # Remove other problematic characters
    gene_name = re.sub(r'[^a-zA-Z0-9\s\-]', '', gene_name)
    # Convert to lowercase
    gene_name = gene_name.lower()
    return gene_name

def preprocess_genedf(genedf):
    for idx, row in genedf.iterrows():
        genelist = row['genelist']
        if isinstance(genelist, list):
            genedf.at[idx, 'genelist'] = [preprocess_gene_name(g) for g in genelist]
        else:
            genedf.at[idx, 'genelist'] = []
        genedf.at[idx, 'Approved symbol'] = preprocess_gene_name(row['Approved symbol'])
    return genedf

def extract_approved_symbols(df_filtered, genedf, list_pmid):
    mg = mygene.MyGeneInfo()
    gene_symbol_map = {}

    # Preprocess genedf and df_filtered
    genedf = preprocess_genedf(genedf)
    df_filtered['Gene_normalized'] = df_filtered['Gene'].apply(lambda x: [preprocess_gene_name(g) for g in x])

    for _, row in df_filtered.iterrows():
        list_gene = row['Gene_normalized']
        pmid = row['pmid']
        gene_symbol_list = []

        print(list_gene)
        print(row['Gene'])

        if pmid in list_pmid:
            for original_gene, normalized_gene in zip(row['Gene'], list_gene):
                # Query MyGene.info for the gene symbol
                try:
                    result = mg.query(normalized_gene, species='human', size=1)
                    if result and 'hits' in result and len(result['hits']) > 0:
                        # Extract the symbol from the result
                        symbol = result['hits'][0].get('symbol', '').lower()
                        print(original_gene,':',symbol)
                        if symbol:
                            print('ok')
                            # Check if the symbol exists in genedf['genelist']
                            matched_genes = genedf[genedf['genelist'].apply(lambda x: symbol in x)]
                            if not matched_genes.empty:
                                print(symbol)
                                print(matched_genes.iloc[0])
                                approved_symbol = matched_genes.iloc[0]['Approved symbol']
                                gene_symbol_list.append(f"{original_gene}:{approved_symbol.upper()}")
                            else:
                                gene_symbol_list.append(f"{original_gene}:")
                        else:
                            print('nall')
                            gene_symbol_list.append(f"{original_gene}:")
                    else:
                        gene_symbol_list.append(f"{original_gene}:")
                except Exception as e:
                    print(f"Error querying gene {normalized_gene}: {e}")
                    gene_symbol_list.append(f"{original_gene}:")

            # Store the gene-approved symbol list in the gene_symbol_map
            gene_symbol_map[pmid] = gene_symbol_list

    # Filter df_filtered to include only rows where pmid is in list_pmid
    final_df = df_filtered[df_filtered['pmid'].isin(list_pmid)].copy()

    # Add the gene_symbol_map as a new column to final_df
    final_df['Gene_approved_symbol'] = final_df['pmid'].apply(lambda pmid: gene_symbol_map.get(pmid, []))

    return final_df

In [61]:
final_df = extract_approved_symbols(df_filtered,genedf, list_pmid)

['rage', 'sodium-glucose cotransporter 2', 'sglt2', 'rage', 'sglt2', 'rage', 'sglt2', 'rage', 'hba1c', 'rage', 'f4 80', 'l-fatty acid binding protein', 'monocyte chemoattractant protein-1', 'intercellular adhesion molecule-1', 'plasminogen activator inhibitor-1', 'transforming growth factor-beta', 'connective tissue growth factor', 'age-rage']
['RAGE', 'sodium-glucose cotransporter 2', 'SGLT2', 'RAGE', 'SGLT2', 'RAGE', 'SGLT2', 'RAGE', 'HbA1c', 'RAGE', 'F4/80', 'L-fatty acid binding protein', 'monocyte chemoattractant protein-1', 'intercellular adhesion molecule-1', 'plasminogen activator inhibitor-1', 'transforming growth factor-beta', 'connective tissue growth factor', 'AGE-RAGE']
RAGE : mok
ok
mok
HGNC ID                                                                 HGNC:9833
Approved symbol                                                               mok
Approved name                                                  MOK protein kinase
Status                                      

In [ ]:
df_49article['pmid'] = df_49article['pmid'].astype(str)
final_df['pmid'] = final_df['pmid'].astype(str)

df_49article_with_symbols = pd.merge(df_49article, final_df.loc[:, ['pmid', 'Gene_approved_symbol']], on='pmid', how='left')

df_49article_with_symbols['Gene_approved_symbol'] = df_49article_with_symbols['Gene_approved_symbol'].apply(lambda x: x if isinstance(x, list) else [])

In [ ]:
df_49article_with_symbols.to_csv('/content/drive/MyDrive/Github/PubMedBERT-NER-Gene/49paper_genes_New2.csv')

#Test Wiki

In [ ]:
!pip install wikipedia-api

In [19]:
import requests
from bs4 import BeautifulSoup

def search_gene_on_wikipedia(gene_name):
    headers = {
        "User-Agent": "GSearch/1.0 (mahdi@gmail.com)"  # Replace with your app name and email
    }

    url = f"https://en.wikipedia.org/wiki/{gene_name}"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        # Parse the Wikipedia page content
        soup = BeautifulSoup(response.content, "html.parser")
        paragraphs = soup.find_all('p')
        content = '\n'.join([p.text for p in paragraphs])
        return content
    else:
        return f"Page not found for gene: {gene_name} (HTTP {response.status_code})"

# Example usage
gene_name = input("Enter a gene name: ")
result = search_gene_on_wikipedia(gene_name)
print(result)

Enter a gene name: RAGE
Rage may refer to:



In [ ]:
!pip install requests lxml

In [18]:
import requests
from lxml import html

def get_wikipedia_title(gene_name):
    headers = {
        "User-Agent": "GSearch/1.0 (mahdi@gmail.com)"  # Replace with your app name and email
    }

    url = f"https://en.wikipedia.org/wiki/{gene_name}"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        # Parse the page content
        tree = html.fromstring(response.content)

        # Extract the title using XPath
        title = tree.xpath('/html/body/div[2]/div/div[3]/main/header/h1/span/text()')

        if title:
            return title[0]
        else:
            return f"Title not found for gene: {gene_name}"
    else:
        return f"Page not found for gene: {gene_name} (HTTP {response.status_code})"

# Example usage
gene_name = input("Enter a gene name: ")
result = get_wikipedia_title(gene_name)
print(result)


Enter a gene name: 	RAGE
Page not found for gene: 	RAGE (HTTP 404)


In [3]:
import requests
from lxml import html

def get_table_cell_link(gene_name):
    headers = {
        "User-Agent": "GSearch/1.0 (mahdi@gmail.com)"  # Replace with your app name and email
    }

    url = f"https://en.wikipedia.org/wiki/{gene_name}"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        # Parse the page content
        tree = html.fromstring(response.content)

        # Extract the anchor tag using XPath
        link_content = tree.xpath('/html/body/div[2]/div/div[3]/main/div[3]/div[3]/div[1]/table/tbody/tr[5]/td/span/a')

        if link_content:
            # Return the link text and href attribute
            link_text = link_content[0].text
            link_href = link_content[0].get('href')
            return f"Link Text: {link_text}, Link URL: {link_href}"
        else:
            return f"No anchor tag found at the specified XPath for gene: {gene_name}"
    else:
        return f"Page not found for gene: {gene_name} (HTTP {response.status_code})"

# Example usage
gene_name = input("Enter a gene name: ")
result = get_table_cell_link(gene_name)
print(result)

Enter a gene name: plasminogen activator
Link Text: PLG, Link URL: https://www.genenames.org/data/gene-symbol-report/#!/hgnc_id/9071


In [21]:
import pandas as pd
import requests
from lxml import html

def get_gene_approved_symbol(gene_name):
    headers = {
        "User-Agent": "GSearch/1.0 (mahdi@gmail.com)"  # Replace with your app name and email
    }

    url = f"https://en.wikipedia.org/wiki/{gene_name} gene"
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        tree = html.fromstring(response.content)
        # Extract the anchor tag using XPath (adjust if needed)
        link_content = tree.xpath('/html/body/div[2]/div/div[3]/main/div[3]/div[3]/div[1]/table/tbody/tr[5]/td/span/a')

        if link_content:
            return link_content[0].text  # Return the link text (Approved symbol)

    return None  # Return None if not found or page doesn't exist

def process_genes(df_filtered, list_pmid):
    final_data = []

    for index, row in df_filtered.iterrows():
        pmid = row['pmid']

        # Check if pmid is in the provided list
        if pmid in list_pmid:
            for gene in row['Gene']:  # Assuming 'Gene' is a list of gene names
                approved_symbol = get_gene_approved_symbol(gene)
                final_data.append({
                    'pmid': pmid,
                    'Gene': gene,
                    'Gene_approved_symbol': f"{gene}: {approved_symbol}" if approved_symbol else f"{gene}: Not Found"
                })

    final_df = pd.DataFrame(final_data)
    return final_df

In [22]:
final_df = process_genes(df_filtered, list_pmid)

In [ ]:
final_df

In [17]:
final_df.loc[final_df['pmid'] == 25611208]

,pmid,Gene,Gene_approved_symbol
433,25611208,RAGE,RAGE: Not Found
434,25611208,sodium-glucose cotransporter 2,sodium-glucose cotransporter 2: Not Found
435,25611208,SGLT2,SGLT2: SLC5A2
436,25611208,RAGE,RAGE: Not Found
437,25611208,SGLT2,SGLT2: SLC5A2
438,25611208,RAGE,RAGE: Not Found
439,25611208,SGLT2,SGLT2: SLC5A2
440,25611208,RAGE,RAGE: Not Found
441,25611208,HbA1c,HbA1c: Not Found
442,25611208,RAGE,RAGE: Not Found
